In [ ]:
%reload_ext dotenv
%dotenv

In [ ]:
from io import BytesIO
from urllib.parse import urlparse

from PIL import Image
from google import genai
from google.genai import types
import replicate
import requests

In [ ]:
def preview(img, size=600):
    p = img.copy()
    p.thumbnail((size, size))
    display(p)

## Wearable on Avatar

In [ ]:
avatar = Image.open("../images/avatars/nimo.jpg")
wearable = Image.open("../images/garments/tops/sweater.jpg")

for im in [avatar, wearable]:
    preview(im, size=400)

In [ ]:
client = genai.Client()

prompt = """
Put this wearable on the avatar.
Preserve the avatar's Sims 3 / PS3 video game art style exactly.
Keep the same pose, background, and all other clothing unchanged.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-image-preview",
    contents=[avatar, wearable, prompt],
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="3:4",
            image_size="512px",
        ),
    ),
)

assert response.parts is not None

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        genai_image = part.as_image()
        assert genai_image is not None
        assert genai_image.image_bytes is not None
        result = Image.open(BytesIO(genai_image.image_bytes))

usage = response.usage_metadata
assert usage is not None
print(
    f"\nToken usage: {usage.prompt_token_count} input, {usage.candidates_token_count} output, {usage.total_token_count} total"
)
print(f"Image size: {result.size} px")
preview(result)

## Masking

In [ ]:
# Run grounded_sam to get a mask of the top wearable on the WOA image
woa_buf = BytesIO()
result.save(woa_buf, format="JPEG")
woa_buf.seek(0)

mask_results = replicate.run(
    "schananas/grounded_sam:ee871c19efb1941f55f66a3d7d960428c8a5afcb77449547fe8e5a3ab9ebc21c",
    input={
        "image": woa_buf,
        "mask_prompt": "sweater",
        "negative_mask_prompt": "",
        "adjustment_factor": 0,
    },
)

mask_image_url = None
for result_url_raw in mask_results:
    result_url_parsed = urlparse(str(result_url_raw))
    if result_url_parsed.path.endswith("/mask.jpg"):
        mask_image_url = result_url_parsed.geturl()
        break

if mask_image_url is None:
    raise ValueError("Could not get mask URL from grounded_sam results")

mask_response = requests.get(mask_image_url)
mask_response.raise_for_status()
mask = Image.open(BytesIO(mask_response.content))
preview(mask)

In [ ]:
# Overlay the mask on the WOA image with a semitransparent red tint
overlay = result.copy().convert("RGBA")
mask_resized = mask.resize(result.size).convert("L")

# Create a red overlay where the mask is white
red = Image.new("RGBA", result.size, (255, 0, 0, 100))
overlay = Image.composite(red, overlay, mask_resized)

preview(overlay)